# Basic

In [ ]:
# install packages

!pip uninstall -y -q sentence-transformers torchvision torchaudio torchtext timm

!pip install -q --no-cache-dir --upgrade --no-deps \
    "transformers==4.51.3" \
    "tokenizers==0.21.1" \
    "huggingface-hub==0.34.4" \
    "safetensors==0.5.3"

!pip install -q tqdm joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 186.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 138.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 249.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 251.2 MB/s eta 0:00:00


In [ ]:
# basic imports
import os
from pathlib import Path
import random

# avoid some unnecessary backend imports
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from google.colab import drive
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModel

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [ ]:
# quick environment check
import importlib.util
import transformers

print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('cuda available:', torch.cuda.is_available())

torch: 2.11.0+cu128
transformers: 4.51.3
cuda available: True


In [ ]:
# experiment settings
seed = 42

embed_batch_size = 8
mlp_batch_size = 512

epochs = 8
lr = 1e-3

encoder_name = 'Qwen/Qwen3-Embedding-0.6B'

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print('device:', device)

device: cuda


In [ ]:
# mount drive and set paths
drive.mount('/content/drive')

project_dir = Path('/content/drive/MyDrive/Colab Notebooks/Project')

hx_dir = project_dir / 'HateXplain' / 'csv_files'
jigsaw_dir = project_dir / 'jigsaw-toxic-comment-classification-challenge'

output_dir = project_dir / 'toxicdetector_style_qwen3_outputs'
output_dir.mkdir(parents=True, exist_ok=True)

print('hatexplain folder:', hx_dir)
print('jigsaw folder:', jigsaw_dir)
print('output folder:', output_dir)

Mounted at /content/drive
hatexplain folder: /content/drive/MyDrive/Colab Notebooks/Project/HateXplain/csv_files
jigsaw folder: /content/drive/MyDrive/Colab Notebooks/Project/jigsaw-toxic-comment-classification-challenge
output folder: /content/drive/MyDrive/Colab Notebooks/Project/toxicdetector_style_qwen3_outputs


# Dataset

In [ ]:
# load hatexplain data
hx_train = pd.read_csv(hx_dir / 'hatexplain_train_full_binary.csv')
hx_val = pd.read_csv(hx_dir / 'hatexplain_validation_full_binary.csv')
hx_test = pd.read_csv(hx_dir / 'hatexplain_test_full_binary.csv')

hx_train = hx_train[['sample_id', 'text', 'label', 'source']].copy()
hx_val = hx_val[['sample_id', 'text', 'label', 'source']].copy()
hx_test = hx_test[['sample_id', 'text', 'label', 'source']].copy()

hx_train['source'] = 'hatexplain'
hx_val['source'] = 'hatexplain'
hx_test['source'] = 'hatexplain'

print('hx_train:', hx_train.shape)
print('hx_val:', hx_val.shape)
print('hx_test:', hx_test.shape)

print(hx_train['label'].value_counts())

hx_train.head()

hx_train: (15383, 4)
hx_val: (1922, 4)
hx_test: (1924, 4)
label
1    9132
0    6251
Name: count, dtype: int64


,sample_id,text,label,source
0,hx_train_0,u really think i would not have been raped by ...,1,hatexplain
1,hx_train_1,the uk has threatened to return radioactive wa...,1,hatexplain
2,hx_train_2,if english is not imposition then hindi is als...,1,hatexplain
3,hx_train_3,no liberal congratulated hindu refugees post c...,1,hatexplain
4,hx_train_4,he said bro even your texts sound redneck what...,1,hatexplain


In [ ]:
# load jigsaw data
jigsaw_path = jigsaw_dir / 'train.csv'

jigsaw_raw = pd.read_csv(jigsaw_path)

print('jigsaw raw:', jigsaw_raw.shape)

jigsaw_raw.head()

jigsaw raw: (159571, 8)


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [ ]:
# convert jigsaw labels into binary labels
toxic_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

jigsaw_raw['label'] = (jigsaw_raw[toxic_cols].sum(axis=1) > 0).astype(int)

jigsaw_df = pd.DataFrame()
jigsaw_df['sample_id'] = ['jg_' + str(i) for i in range(len(jigsaw_raw))]
jigsaw_df['text'] = jigsaw_raw['comment_text'].astype(str)
jigsaw_df['label'] = jigsaw_raw['label'].astype(int)
jigsaw_df['source'] = 'jigsaw'

print('jigsaw:', jigsaw_df.shape)
print(jigsaw_df['label'].value_counts())

jigsaw_df.head()

jigsaw: (159571, 4)
label
0    143346
1     16225
Name: count, dtype: int64


,sample_id,text,label,source
0,jg_0,Explanation\nWhy the edits made under my usern...,0,jigsaw
1,jg_1,D'aww! He matches this background colour I'm s...,0,jigsaw
2,jg_2,"Hey man, I'm really not trying to edit war. It...",0,jigsaw
3,jg_3,"""\nMore\nI can't make any real suggestions on ...",0,jigsaw
4,jg_4,"You, sir, are my hero. Any chance you remember...",0,jigsaw


In [ ]:
# split jigsaw train.csv by myself
jigsaw_train, jigsaw_temp = train_test_split(
    jigsaw_df,
    test_size=0.3,
    random_state=seed,
    stratify=jigsaw_df['label']
)

jigsaw_val, jigsaw_test = train_test_split(
    jigsaw_temp,
    test_size=0.5,
    random_state=seed,
    stratify=jigsaw_temp['label']
)

print('jigsaw_train:', jigsaw_train.shape)
print('jigsaw_val:', jigsaw_val.shape)
print('jigsaw_test:', jigsaw_test.shape)

print(jigsaw_train['label'].value_counts())

jigsaw_train: (111699, 4)
jigsaw_val: (23936, 4)
jigsaw_test: (23936, 4)
label
0    100342
1     11357
Name: count, dtype: int64


In [ ]:
# combine hatexplain and jigsaw
train_df = pd.concat([hx_train, jigsaw_train], axis=0).reset_index(drop=True)
val_df = pd.concat([hx_val, jigsaw_val], axis=0).reset_index(drop=True)
test_df = pd.concat([hx_test, jigsaw_test], axis=0).reset_index(drop=True)

train_df = train_df.sample(frac=1, random_state=seed).reset_index(drop=True)
val_df = val_df.sample(frac=1, random_state=seed).reset_index(drop=True)
test_df = test_df.sample(frac=1, random_state=seed).reset_index(drop=True)

train_df = train_df.dropna(subset=['text', 'label']).reset_index(drop=True)
val_df = val_df.dropna(subset=['text', 'label']).reset_index(drop=True)
test_df = test_df.dropna(subset=['text', 'label']).reset_index(drop=True)

train_df['text'] = train_df['text'].astype(str)
val_df['text'] = val_df['text'].astype(str)
test_df['text'] = test_df['text'].astype(str)

print('train:', train_df.shape)
print('val:', val_df.shape)
print('test:', test_df.shape)

print('\ntrain label count')
print(train_df['label'].value_counts())

print('\ntrain source count')
print(train_df['source'].value_counts())

train: (127082, 4)
val: (25858, 4)
test: (25860, 4)

train label count
label
0    106593
1     20489
Name: count, dtype: int64

train source count
source
jigsaw        111699
hatexplain     15383
Name: count, dtype: int64


In [ ]:
# save common data for later notebooks
train_df.to_csv(output_dir / 'common_train_hx_jigsaw.csv', index=False)
val_df.to_csv(output_dir / 'common_val_hx_jigsaw.csv', index=False)
test_df.to_csv(output_dir / 'common_test_hx_jigsaw.csv', index=False)

print('saved common data')

saved common data


# ToxicDetector-style representation

In [ ]:
# load qwen3 embedding model
# this replaces the inaccessible llama-2 representation in the original ToxicDetector

tokenizer = AutoTokenizer.from_pretrained(
    encoder_name,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = 'left'

model_dtype = torch.float16 if device == 'cuda' else torch.float32

text_encoder = AutoModel.from_pretrained(
    encoder_name,
    torch_dtype=model_dtype,
    trust_remote_code=True
)

text_encoder = text_encoder.to(device)
text_encoder.eval()

max_len = 512

print('loaded:', encoder_name)
print('device:', device)
print('max length:', max_len)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

loaded: Qwen/Qwen3-Embedding-0.6B
device: cuda
max length: 512


In [ ]:
# task instruction for qwen embedding
task_inst = 'Instruct: Represent this online comment for toxic content detection.\nQuery: '


def add_inst(text):
    return task_inst + str(text)

In [ ]:
# pooling function
def last_token_pool(last_hidden, attention_mask):
    left_padding = attention_mask[:, -1].sum() == attention_mask.shape[0]

    if left_padding:
        return last_hidden[:, -1]

    seq_lens = attention_mask.sum(dim=1) - 1
    batch_size = last_hidden.shape[0]
    batch_ids = torch.arange(batch_size, device=last_hidden.device)

    return last_hidden[batch_ids, seq_lens]

In [ ]:
# encode texts into normalized embeddings
def encode_texts(text_list, batch_size=8):
    all_vecs = []

    for start in tqdm(range(0, len(text_list), batch_size)):
        batch_texts = text_list[start:start + batch_size]

        batch_tokens = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors='pt'
        )

        batch_tokens = {
            k: v.to(device)
            for k, v in batch_tokens.items()
        }

        with torch.no_grad():
            outputs = text_encoder(**batch_tokens)

            vecs = last_token_pool(
                outputs.last_hidden_state,
                batch_tokens['attention_mask']
            )

            vecs = F.normalize(vecs, p=2, dim=1)

        all_vecs.append(vecs.float().cpu().numpy())

    all_vecs = np.concatenate(all_vecs, axis=0)

    return all_vecs.astype('float32')

In [ ]:
# toxic concept prompts
toxic_prompts = [
    'This text contains hate speech or attacks a group of people.',
    'This text contains offensive or abusive language.',
    'This text insults or humiliates someone.',
    'This text contains harassment or threat.',
    'This text is toxic in an online comment section.',
    'This text expresses identity-based hate or discrimination.'
]

concept_inputs = []

for text in toxic_prompts:
    concept_inputs.append(add_inst(text))

concept_vecs = encode_texts(
    concept_inputs,
    batch_size=4
)

concept_mean = concept_vecs.mean(axis=0)
concept_mean = concept_mean / np.linalg.norm(concept_mean)
concept_mean = concept_mean.astype('float32')

print('concept_vecs:', concept_vecs.shape)
print('concept_mean:', concept_mean.shape)

  0%|          | 0/2 [00:00<?, ?it/s]

concept_vecs: (6, 1024)
concept_mean: (1024,)


In [ ]:
# make ToxicDetector-style interaction features
def make_features(text_list):
    input_list = []

    for text in text_list:
        input_list.append(add_inst(text))

    text_vecs = encode_texts(
        input_list,
        batch_size=embed_batch_size
    )

    sim_all = np.matmul(text_vecs, concept_vecs.T)
    sim_all = sim_all.astype('float32')

    sim_mean = sim_all.mean(axis=1).reshape(-1, 1).astype('float32')
    sim_max = sim_all.max(axis=1).reshape(-1, 1).astype('float32')

    concept_big = concept_mean.reshape(1, -1)

    product_vecs = text_vecs * concept_big
    abs_vecs = np.abs(text_vecs - concept_big)

    features = np.concatenate(
        [text_vecs, product_vecs, abs_vecs, sim_all, sim_mean, sim_max],
        axis=1
    )

    return features.astype('float32')

In [ ]:
# make or load features
# qwen encoding takes time, so I save each split

x_train_path = output_dir / 'qwen3_x_train.npy'
y_train_path = output_dir / 'qwen3_y_train.npy'

x_val_path = output_dir / 'qwen3_x_val.npy'
y_val_path = output_dir / 'qwen3_y_val.npy'

x_test_path = output_dir / 'qwen3_x_test.npy'
y_test_path = output_dir / 'qwen3_y_test.npy'


def load_or_make(df, x_path, y_path, name):
    if x_path.exists() and y_path.exists():
        print('load saved features:', name)

        x = np.load(x_path)
        y = np.load(y_path)

    else:
        print('make new features:', name)

        x = make_features(df['text'].tolist())
        y = df['label'].astype(int).values

        np.save(x_path, x)
        np.save(y_path, y)

    print(name, x.shape)

    return x, y


x_train, y_train = load_or_make(train_df, x_train_path, y_train_path, 'train')
x_val, y_val = load_or_make(val_df, x_val_path, y_val_path, 'val')
x_test, y_test = load_or_make(test_df, x_test_path, y_test_path, 'test')

make new features: train


  0%|          | 0/15886 [00:00<?, ?it/s]

train (127082, 3080)
make new features: val


  0%|          | 0/3233 [00:00<?, ?it/s]

val (25858, 3080)
make new features: test


  0%|          | 0/3233 [00:00<?, ?it/s]

test (25860, 3080)


# MLP

In [ ]:
# simple mlp classifier
class ToxicMlp(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 1)
        )

    def forward(self, x):
        out = self.net(x)
        return out.squeeze(1)

In [ ]:
# prepare pytorch data
from torch.utils.data import TensorDataset, DataLoader

x_train_tensor = torch.from_numpy(x_train)
y_train_tensor = torch.from_numpy(y_train.astype('float32'))

train_data = TensorDataset(x_train_tensor, y_train_tensor)

train_loader = DataLoader(
    train_data,
    batch_size=mlp_batch_size,
    shuffle=True
)

input_dim = x_train.shape[1]

td_model = ToxicMlp(input_dim).to(device)

pos_num = (y_train == 1).sum()
neg_num = (y_train == 0).sum()

pos_weight_value = neg_num / pos_num
pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32).to(device)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(td_model.parameters(), lr=lr)

print('input_dim:', input_dim)
print('positive samples:', pos_num)
print('negative samples:', neg_num)
print('pos_weight:', pos_weight_value)

input_dim: 3080
positive samples: 20489
negative samples: 106593
pos_weight: 5.2024500951730195


In [ ]:
# prediction helper
def predict_scores(model, x, batch_size=1024):
    model.eval()

    score_list = []
    x_tensor = torch.from_numpy(x)

    with torch.no_grad():
        for start in range(0, len(x_tensor), batch_size):
            batch_x = x_tensor[start:start + batch_size].to(device)

            logits = model(batch_x)
            probs = torch.sigmoid(logits)

            score_list.append(probs.cpu().numpy())

    scores = np.concatenate(score_list, axis=0)

    return scores

In [ ]:
# choose threshold by validation f1
def find_best_threshold(y_true, scores):
    best_t = 0.5
    best_f1 = 0

    for t in np.arange(0.1, 0.91, 0.05):
        preds = (scores >= t).astype(int)
        now_f1 = f1_score(y_true, preds, zero_division=0)

        if now_f1 > best_f1:
            best_f1 = now_f1
            best_t = t

    return best_t, best_f1

In [ ]:
# train mlp
best_f1 = 0
best_t = 0.5
best_epoch = 0
best_state = None

history_rows = []

for epoch in range(epochs):
    td_model.train()

    total_loss = 0

    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()

        logits = td_model(batch_x)
        loss = loss_fn(logits, batch_y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(batch_y)

    avg_loss = total_loss / len(train_data)

    val_score = predict_scores(td_model, x_val)
    now_t, now_f1 = find_best_threshold(y_val, val_score)

    row = {
        'epoch': epoch + 1,
        'train_loss': avg_loss,
        'val_f1': now_f1,
        'threshold': now_t
    }

    history_rows.append(row)

    print(
        'epoch:', epoch + 1,
        'loss:', round(avg_loss, 4),
        'val_f1:', round(now_f1, 4),
        't:', now_t
    )

    if now_f1 > best_f1:
        best_f1 = now_f1
        best_t = now_t
        best_epoch = epoch + 1

        best_state = {}

        for k, v in td_model.state_dict().items():
            best_state[k] = v.detach().cpu().clone()

history_df = pd.DataFrame(history_rows)
history_df.to_csv(output_dir / 'training_history.csv', index=False)

td_model.load_state_dict(best_state)
td_model = td_model.to(device)

print('best epoch:', best_epoch)
print('best threshold:', best_t)
print('best val f1:', best_f1)

epoch: 1 loss: 0.4747 val_f1: 0.7657 t: 0.7500000000000002
epoch: 2 loss: 0.3979 val_f1: 0.7684 t: 0.8500000000000002
epoch: 3 loss: 0.3852 val_f1: 0.7802 t: 0.8000000000000002
epoch: 4 loss: 0.3693 val_f1: 0.7798 t: 0.8000000000000002
epoch: 5 loss: 0.363 val_f1: 0.7789 t: 0.8500000000000002
epoch: 6 loss: 0.3553 val_f1: 0.7829 t: 0.8000000000000002
epoch: 7 loss: 0.3506 val_f1: 0.7821 t: 0.8500000000000002
epoch: 8 loss: 0.3438 val_f1: 0.7858 t: 0.6500000000000001
best epoch: 8
best threshold: 0.6500000000000001
best val f1: 0.7857823777747515


# Result

In [ ]:
# metric function
def get_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'false_positive_rate': fpr,
        'false_negative_rate': fnr,
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp)
    }

In [ ]:
# test set prediction
test_score = predict_scores(td_model, x_test)
test_pred = (test_score >= best_t).astype(int)

test_out = test_df.copy()
test_out['td_score'] = test_score
test_out['td_pred'] = test_pred

rows = []

mixed_metrics = get_metrics(
    test_out['label'].astype(int).values,
    test_out['td_pred'].astype(int).values
)

mixed_row = {'method': 'toxicdetector_style_qwen3_mlp_mixed'}
mixed_row.update(mixed_metrics)

rows.append(mixed_row)

for src in ['hatexplain', 'jigsaw']:
    part_df = test_out[test_out['source'] == src].copy()

    part_metrics = get_metrics(
        part_df['label'].astype(int).values,
        part_df['td_pred'].astype(int).values
    )

    part_row = {'method': 'toxicdetector_style_qwen3_mlp_' + src}
    part_row.update(part_metrics)

    rows.append(part_row)

result_df = pd.DataFrame(rows)

result_df

,method,accuracy,precision,recall,f1,false_positive_rate,false_negative_rate,tn,fp,fn,tp
0,toxicdetector_style_qwen3_mlp_mixed,0.941454,0.774348,0.813758,0.793564,0.038054,0.186242,21436,848,666,2910
1,toxicdetector_style_qwen3_mlp_hatexplain,0.717256,0.703956,0.903678,0.791411,0.554987,0.096322,348,434,110,1032
2,toxicdetector_style_qwen3_mlp_jigsaw,0.959475,0.819372,0.771569,0.794752,0.019254,0.228431,21088,414,556,1878


In [ ]:
# save outputs for later fusion
save_cols = ['sample_id', 'text', 'label', 'source', 'td_score', 'td_pred']

td_out = test_out[save_cols].copy()

td_out.to_csv(output_dir / 'toxicdetector_style_outputs.csv', index=False)
result_df.to_csv(output_dir / 'toxicdetector_style_metrics.csv', index=False)

print('saved:')
print(output_dir / 'toxicdetector_style_outputs.csv')
print(output_dir / 'toxicdetector_style_metrics.csv')

saved:
/content/drive/MyDrive/Colab Notebooks/Project/toxicdetector_style_qwen3_outputs/toxicdetector_style_outputs.csv
/content/drive/MyDrive/Colab Notebooks/Project/toxicdetector_style_qwen3_outputs/toxicdetector_style_metrics.csv


In [ ]:
# save model
model_state = {}

for k, v in td_model.state_dict().items():
    model_state[k] = v.detach().cpu()

model_pack = {
    'model_state': model_state,
    'input_dim': input_dim,
    'best_threshold': float(best_t),
    'encoder_name': encoder_name,
    'toxic_prompts': toxic_prompts,
    'task_instruction': task_inst,
    'max_len': max_len
}

torch.save(model_pack, output_dir / 'toxicdetector_style_qwen3_mlp.pt')

print('saved model')

saved model


In [ ]:
# save setting
setting_df = pd.DataFrame({
    'item': [
        'encoder_name',
        'classifier',
        'best_epoch',
        'best_threshold',
        'feature_type',
        'loss',
        'optimizer'
    ],
    'value': [
        encoder_name,
        'small MLP: Linear 512 + Linear 128 + dropout',
        str(best_epoch),
        str(best_t),
        'text_vec + product_vec + abs_vec + concept similarities',
        'BCEWithLogitsLoss with positive class weight',
        'AdamW'
    ]
})

setting_df.to_csv(output_dir / 'toxicdetector_style_setting.csv', index=False)

setting_df

,item,value
0,encoder_name,Qwen/Qwen3-Embedding-0.6B
1,classifier,small MLP: Linear 512 + Linear 128 + dropout
2,best_epoch,8
3,best_threshold,0.6500000000000001
4,feature_type,text_vec + product_vec + abs_vec + concept sim...
5,loss,BCEWithLogitsLoss with positive class weight
6,optimizer,AdamW


# Supplementary

In [ ]:
# uncertain cases for confidence fusion
uncertain_df = td_out[
    (td_out['td_score'] > 0.3) & (td_out['td_score'] < 0.7)
].copy()

print('uncertain cases:', uncertain_df.shape)
print(uncertain_df['source'].value_counts())

uncertain_df.to_csv(output_dir / 'toxicdetector_style_uncertain_cases.csv', index=False)

uncertain_df.head(10)

uncertain cases: (2058, 6)
source
jigsaw        1605
hatexplain     453
Name: count, dtype: int64


,sample_id,text,label,source,td_score,td_pred
26,jg_83813,this guy nailed it on the head! Paralympiakos ...,0,jigsaw,0.303830,0
49,jg_48310,"listen here buddy, this kyle. I think my cool...",0,jigsaw,0.618368,0
58,jg_105961,It's true... \n\n.. the incompetent are very s...,0,jigsaw,0.497225,0
70,jg_58472,"""\n\nSOME USERS CAN DO WHATEVER THEY WANT, WHI...",1,jigsaw,0.476408,0
72,jg_77390,Warning \n\nIf you continued imposing wrong re...,1,jigsaw,0.588183,0
102,jg_7145,SuPeRTR0LL WiLL LiVe FoReVeR!\niF You DoN'T Re...,1,jigsaw,0.485938,0
105,jg_108766,"""\nUnfunny. (talk to me) """,0,jigsaw,0.311671,0
139,jg_62455,"Images \n\nInsulting images need to stop, now....",0,jigsaw,0.523164,0
142,jg_27353,It is amusing to see someone complain of censo...,0,jigsaw,0.454487,0
150,jg_60815,"""::::Hahaha tell Cinema to cry me a river. Yo...",0,jigsaw,0.502470,0


In [ ]:
# quick checks
print('score summary')
print(td_out['td_score'].describe())

print('\nprediction count')
print(td_out['td_pred'].value_counts())

print('\nsource count')
print(td_out['source'].value_counts())

score summary
count    2.586000e+04
mean     1.786504e-01
std      3.163304e-01
min      4.974837e-08
25%      5.560560e-04
50%      6.009092e-03
75%      1.606378e-01
max      9.997804e-01
Name: td_score, dtype: float64

prediction count
td_pred
0    22102
1     3758
Name: count, dtype: int64

source count
source
jigsaw        23936
hatexplain     1924
Name: count, dtype: int64


# save validation outputs after reopening notebook

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

project_dir = Path('/content/drive/MyDrive/Colab Notebooks/Project')
output_dir = project_dir / 'toxicdetector_style_qwen3_outputs'

for file in output_dir.iterdir():
    print(file.name)

common_train_hx_jigsaw.csv
common_val_hx_jigsaw.csv
common_test_hx_jigsaw.csv
qwen3_x_train.npy
qwen3_y_train.npy
qwen3_x_val.npy
qwen3_y_val.npy
qwen3_x_test.npy
qwen3_y_test.npy
training_history.csv
toxicdetector_style_outputs.csv
toxicdetector_style_metrics.csv
toxicdetector_style_qwen3_mlp.pt
toxicdetector_style_setting.csv
toxicdetector_style_uncertain_cases.csv


In [ ]:
# save validation outputs after reopening notebook

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

project_dir = Path('/content/drive/MyDrive/Colab Notebooks/Project')
output_dir = project_dir / 'toxicdetector_style_qwen3_outputs'

device = 'cuda' if torch.cuda.is_available() else 'cpu'


class ToxicMlp(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 1)
        )

    def forward(self, x):
        out = self.net(x)
        return out.squeeze(1)


def predict_scores(model, x, batch_size=1024):
    model.eval()

    score_list = []
    x_tensor = torch.from_numpy(x)

    with torch.no_grad():
        for start in range(0, len(x_tensor), batch_size):
            batch_x = x_tensor[start:start + batch_size].to(device)

            logits = model(batch_x)
            probs = torch.sigmoid(logits)

            score_list.append(probs.cpu().numpy())

    scores = np.concatenate(score_list, axis=0)

    return scores


# load saved validation data and model
val_df = pd.read_csv(output_dir / 'common_val_hx_jigsaw.csv')

x_val = np.load(output_dir / 'qwen3_x_val.npy')
y_val = np.load(output_dir / 'qwen3_y_val.npy')

model_pack = torch.load(
    output_dir / 'toxicdetector_style_qwen3_mlp.pt',
    map_location=device
)

input_dim = model_pack['input_dim']
best_t = model_pack['best_threshold']

td_model = ToxicMlp(input_dim).to(device)
td_model.load_state_dict(model_pack['model_state'])
td_model.eval()

print('loaded model')
print('input_dim:', input_dim)
print('best threshold:', best_t)
print('x_val:', x_val.shape)

loaded model
input_dim: 3080
best threshold: 0.6500000000000001
x_val: (25858, 3080)


In [ ]:
# make validation prediction and save

val_score = predict_scores(td_model, x_val)
val_pred = (val_score >= best_t).astype(int)

val_out = val_df.copy()
val_out['td_score'] = val_score
val_out['td_pred'] = val_pred

save_cols = ['sample_id', 'text', 'label', 'source', 'td_score', 'td_pred']

td_val_out = val_out[save_cols].copy()

td_val_out.to_csv(output_dir / 'toxicdetector_style_val_outputs.csv', index=False)

print('saved validation outputs:')
print(output_dir / 'toxicdetector_style_val_outputs.csv')

td_val_out.head()

saved validation outputs:
/content/drive/MyDrive/Colab Notebooks/Project/toxicdetector_style_qwen3_outputs/toxicdetector_style_val_outputs.csv


,sample_id,text,label,source,td_score,td_pred
0,jg_36797,"""\n\nShall you immediately stop this sock-pupp...",0,jigsaw,0.121980,0
1,hx_validation_341,this is what <number> years of kike brainwashi...,1,hatexplain,0.927676,1
2,hx_validation_1510,non asians btw u cant make jokes like “ why do...,0,hatexplain,0.631207,0
3,jg_120449,I don't really think renaming should be a cent...,0,jigsaw,0.000073,0
4,jg_112467,Apologies \n\nfor the stupid pointless bickeri...,0,jigsaw,0.047021,0
